# Hindcast 0008-01: RMSE, EPFlux Components, And Spread Timing

拆自 `Hindcast_0008_01_tropos_wave_tests.ipynb` 的 RMSE/EPFlux 主线 block。图会写入按 notebook 和内容拆分的子目录。

In [ ]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import linregress, pearsonr

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
})

CASE = "0008-01"
REF_YEAR = 8

REPO_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
TEST_ROOT = REPO_ROOT / "Hindcast_experiment" / "TEST_TROPOS"
OUT_ROOT = TEST_ROOT / "outputs" / CASE
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
HINDCAST_ROOT = DATA_ROOT / "Hindcast"
BWCN_ROOT = DATA_ROOT / "BWCN"
B2000_ROOT = DATA_ROOT / "B2000WCN001002_timefixed"

CASE_ROOT = HINDCAST_ROOT / CASE

# Main windows. The EP window matches the old Hindcast_vertical_analysis scatter.
O3_END = (5, 30)
EP_WINDOW = ((1, 20), (2, 10))
EP_WINDOW_LABEL = "Jan20-Feb10"
AO_BASE_WINDOW = EP_WINDOW
AO_BASE_WINDOW_LABEL = EP_WINDOW_LABEL
AO_TEST_WINDOWS = {
    "Jan20-Feb10": EP_WINDOW,
    "Jan": ((1, 1), (1, 31)),
    "Jan-Feb": ((1, 1), (2, 28)),
    "Jan-Mar": ((1, 1), (3, 31)),
}
Z300_WINDOW = EP_WINDOW
Z300_WINDOW_LABEL = EP_WINDOW_LABEL
MONTH_WINDOWS = {
    "Jan": ((1, 1), (1, 31)),
    "Feb": ((2, 1), (2, 28)),
    "Mar": ((3, 1), (3, 31)),
    "Apr": ((4, 1), (4, 30)),
    "May": ((5, 1), (5, 30)),
}
MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May"]

LAT_EP = (40.0, 80.0)
LAT_POLAR = (60.0, 90.0)
LAT_Z300 = (20.0, 90.0)
PLEV_EP_PA = 10000.0
PLEV_EP_HPA = PLEV_EP_PA / 100.0
PLEV_U_PA = 1000.0
PLEV_Z300_PA = 30000.0
PLEV_Z300_HPA = PLEV_Z300_PA / 100.0

EP_SCALAR_DESCRIPTION = (
    f"mean -ep2 vertical EP-flux component at {PLEV_EP_HPA:.0f} hPa, "
    f"cos-lat mean {LAT_EP[0]:.0f}-{LAT_EP[1]:.0f}N, {EP_WINDOW_LABEL}; not EP-flux divergence"
)
Z300_PATTERN_DESCRIPTION = (
    f"{PLEV_Z300_HPA:.0f} hPa height, {Z300_WINDOW_LABEL} mean, "
    f"{LAT_Z300[0]:.0f}-{LAT_Z300[1]:.0f}N, zonal mean removed before pattern metrics"
)

WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
WAVE_LABELS = {
    "all_waves": "All waves",
    "wave1": "Wave 1",
    "wave2": "Wave 2",
    "wave_rest": "Wave rest",
    "wave1_plus_wave2": "Wave 1 + Wave 2",
}

# Expensive sections cache their results. Keep True for the full requested test.
RUN_Z300_DIAGNOSTICS = True
BUILD_U60N10_IF_MISSING = True
CLIM_MAX_B2000_YEARS_FOR_Z300 = None  # None = all B2000WCN001002 Z3 years available.

print(f"Output root: {OUT_ROOT}")
print(f"Case root exists: {CASE_ROOT.exists()} -> {CASE_ROOT}")

In [ ]:
# -----------------------------
# Shared helpers
# -----------------------------

def member_short_id(member) -> str:
    text = str(member)
    m = re.search(r"\.(\d{3})\.cam\.h3", text)
    if m:
        return m.group(1)
    m = re.search(r"\.(\d{3})\.", text)
    return m.group(1) if m else text


def date_parts(date_values):
    arr = np.asarray(date_values, dtype=np.int64)
    year = arr // 10000
    mmdd = arr % 10000
    month = mmdd // 100
    day = mmdd % 100
    return year, month, day


def date_mask(date_values, start=(1, 1), end=(5, 30), year: Optional[int] = None):
    yy, mm, dd = date_parts(date_values)
    start_key = start[0] * 100 + start[1]
    end_key = end[0] * 100 + end[1]
    key = mm * 100 + dd
    mask = (key >= start_key) & (key <= end_key)
    if year is not None:
        mask = mask & (yy == int(year))
    return mask


def one_dim_date(ds_or_da) -> np.ndarray:
    date = ds_or_da["date"]
    if "member" in date.dims:
        date = date.isel(member=0)
    return np.asarray(date.values, dtype=np.int64)


def assign_member_short_coord(da: xr.DataArray) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    mids = [member_short_id(v) for v in da["member"].values]
    return da.assign_coords(member_short=("member", mids))


def select_latband(da: xr.DataArray, lat_range: Tuple[float, float], lat_name="lat") -> xr.DataArray:
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    lo, hi = lat_range
    return da.sel({lat_name: slice(hi, lo) if descending else slice(lo, hi)})


def coslat_mean(da: xr.DataArray, lat_range: Optional[Tuple[float, float]] = None, lat_name="lat") -> xr.DataArray:
    if lat_range is not None:
        da = select_latband(da, lat_range, lat_name=lat_name)
    weights = np.cos(np.deg2rad(da[lat_name])).clip(0, 1)
    return da.weighted(weights.fillna(0)).mean(lat_name, skipna=True)


def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = pearsonr(x[mask], y[mask])
    return float(r), float(p)


def scatter_fit(ax, df, xcol, ycol, title, xlabel=None, ylabel=None, color="tab:blue", annotate_members=True):
    sub = df[[xcol, ycol, "member"]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(sub[xcol], sub[ycol], s=62, color=color, edgecolor="black", linewidth=0.5, alpha=0.88)
    if annotate_members:
        for _, row in sub.iterrows():
            ax.text(row[xcol], row[ycol], str(row["member"]), fontsize=7, alpha=0.65)
    if len(sub) >= 3:
        fit = linregress(sub[xcol].values, sub[ycol].values)
        xx = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(xx, fit.slope * xx + fit.intercept, color="crimson", ls="--", lw=1.8)
        ax.text(
            0.03, 0.97,
            f"R = {fit.rvalue:.3f}\nP = {fit.pvalue:.2e}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.84, edgecolor="0.7"),
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    return ax


def savefig(fig, name):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    return png, pdf


def mmdd_label(date_values):
    _, mm, dd = date_parts(date_values)
    return np.array([f"{int(m):02d}-{int(d):02d}" for m, d in zip(mm, dd)])


def month_ticks(date_values):
    _, mm, dd = date_parts(date_values)
    positions, labels = [], []
    names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}
    seen = set()
    for i, (m, d) in enumerate(zip(mm, dd)):
        if int(d) == 1 and int(m) not in seen:
            positions.append(i)
            labels.append(names.get(int(m), str(int(m))))
            seen.add(int(m))
    return positions, labels


def standardize_1d(y):
    y = np.asarray(y, dtype=float)
    return (y - np.nanmean(y)) / np.nanstd(y)

In [ ]:
# -----------------------------
# Data loaders for the cleaned Hindcast products
# -----------------------------

def load_hindcast_o3(case=CASE, pressure_tag="30_70hPa"):
    path = HINDCAST_ROOT / case / "partial_O3" / f"{case}_partial_O3_all_ranges_members.nc"
    if not path.exists():
        raise FileNotFoundError(f"Missing cleaned partial O3 product: {path}")
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        da = assign_member_short_coord(ds[var]).load()
        date = one_dim_date(ds)
    mask = date_mask(date, start=(1, 1), end=O3_END)
    da = da.isel(lead_time=mask)
    date = date[mask]
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_bwcn_ref_o3(year=REF_YEAR, pressure_tag="30_70hPa"):
    path = BWCN_ROOT / "partial_O3" / "BWCN_partial_O3_all_ranges.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=(1, 1), end=O3_END, year=year)
        da = ds[var].isel(time=mask).load()
        date = date[mask]
    da = da.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return da, date


def load_epflux_wave(case=CASE, wave="all_waves", plev_pa=PLEV_EP_PA, lat_range=LAT_EP):
    path = HINDCAST_ROOT / case / "EPflux_daily_ubar" / wave / f"EPFLUX_{wave}_{case}_members_time_plev_lat.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        ep2 = assign_member_short_coord(ds["ep2"])
        ep2_100 = ep2.sel(plev=plev_pa, method="nearest")
        ep2_100 = coslat_mean(ep2_100, lat_range=lat_range)
        # Use -ep2 so positive means upward wave activity, matching the old Fz scatter convention.
        out = (-ep2_100).load()
    return out


def ep_window_mean(ep_da: xr.DataArray, date_values, start_end=EP_WINDOW):
    start, end = start_end
    mask = date_mask(date_values, start=start, end=end)
    return ep_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def load_all_wave_metrics(case=CASE, date_values=None, start_end=EP_WINDOW):
    if date_values is None:
        _, date_values = load_hindcast_o3(case)
    rows = []
    series = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_values)))
        metric = ep_window_mean(da, date_values, start_end=start_end)
        series[wave] = da.assign_coords(date=("lead_time", date_values[: da.sizes["lead_time"]]))
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    return pd.DataFrame(rows), series


def compute_rmse_table(o3_hind: xr.DataArray, date_h, o3_ref: xr.DataArray, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30"):
    mh = date_mask(date_h, start=start, end=end)
    mr = date_mask(date_ref, start=start, end=end, year=REF_YEAR)
    h = o3_hind.isel(lead_time=mh)
    r = o3_ref.isel(lead_time=mr)
    n = min(h.sizes["lead_time"], r.sizes["lead_time"])
    h = h.isel(lead_time=slice(0, n))
    r = r.isel(lead_time=slice(0, n))
    diff = h - r
    rmse = np.sqrt((diff ** 2).mean("lead_time", skipna=True))
    return pd.DataFrame({
        "member": [str(v) for v in rmse["member_short"].values],
        "RMSE_DU": rmse.values.astype(float),
        "rmse_window": label,
        "n_days": n,
    }).sort_values("RMSE_DU").reset_index(drop=True)


def merge_rmse_ep(rmse_df, ep_metric_df):
    wide = ep_metric_df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    return rmse_df.merge(wide, on="member", how="left")

In [ ]:
# -----------------------------
# Product sanity check for 0008-01
# -----------------------------
expected = {
    "partial_O3": CASE_ROOT / "partial_O3" / f"{CASE}_partial_O3_all_ranges_members.nc",
    "EP all": CASE_ROOT / "EPflux_daily_ubar" / "all_waves" / f"EPFLUX_all_waves_{CASE}_members_time_plev_lat.nc",
    "EP wave1": CASE_ROOT / "EPflux_daily_ubar" / "wave1" / f"EPFLUX_wave1_{CASE}_members_time_plev_lat.nc",
    "EP wave2": CASE_ROOT / "EPflux_daily_ubar" / "wave2" / f"EPFLUX_wave2_{CASE}_members_time_plev_lat.nc",
    "EP rest": CASE_ROOT / "EPflux_daily_ubar" / "wave_rest" / f"EPFLUX_wave_rest_{CASE}_members_time_plev_lat.nc",
    "FWD": CASE_ROOT / "final_warming_date" / f"{CASE}_FWD_plev_member.nc",
    "AO/NAM projection": CASE_ROOT / "NAM_B2000WCN_projection" / f"{CASE}_AO_NAM_B2000WCN_projection_members.nc",
}
for name, path in expected.items():
    print(f"{name:18s}: {path.exists()}  {path}")

with xr.open_dataset(expected["partial_O3"], decode_times=False) as ds:
    print("\npartial_O3 dims:", dict(ds.sizes))
    print("partial_O3 vars:", list(ds.data_vars))
with xr.open_dataset(expected["EP all"], decode_times=False) as ds:
    print("\nEP all dims:", dict(ds.sizes))
    print("EP attrs:", {k: ds.attrs.get(k) for k in ["method", "do_ubar", "use_omega_w_correction"]})

In [ ]:
# -----------------------------
# Figure routing for the split notebook
# -----------------------------
NOTEBOOK_STEM = "Hindcast_0008_01_01_rmse_epflux_spread"

def use_figure_dir(content_tag: str):
    global FIG_DIR
    FIG_DIR = OUT_ROOT / "figures" / NOTEBOOK_STEM / content_tag
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print("Figure dir:", FIG_DIR)
    return FIG_DIR


## 图：O3 RMSE 与早期 EPFlux

**做什么**：复现 cleaned Hindcast 数据中的 O3 RMSE vs 早期波活动散点图。

**怎么做**：O3 RMSE 使用 `60-90N, 30-70 hPa` partial O3，窗口为 Jan1-May30，对比 BWCN 第 0008 年。EPFlux 指标为 `EP100 = mean(-ep2)`，即 100 hPa、40-80N、Jan20-Feb10 平均的垂直 EP flux 分量；不是 divergence。

**科学问题**：早期进入平流层的波活动强弱，是否能解释后续臭氧柱偏离参考年的程度？

**预期**：如果波活动控制了可预报性，EP100 越强的成员应有更大的 O3 RMSE。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("01_rmse_vs_epflux")


In [ ]:
# -----------------------------
# Reproduce O3 RMSE vs winter wave activity with cleaned data
# -----------------------------
o3_h, date_h = load_hindcast_o3(CASE)
o3_ref, date_ref = load_bwcn_ref_o3(REF_YEAR)

ep_metric_df, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)
rmse_all = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30")
rmse_ep_all = merge_rmse_ep(rmse_all, ep_metric_df)
rmse_ep_all.to_csv(TABLE_DIR / f"{CASE}_rmse_epflux_wave_metrics_Jan20_Feb10.csv", index=False)

print(rmse_ep_all.head())
print("\nRMSE range:", rmse_ep_all["RMSE_DU"].min(), rmse_ep_all["RMSE_DU"].max())
print("Saved table:", TABLE_DIR / f"{CASE}_rmse_epflux_wave_metrics_Jan20_Feb10.csv")

fig, ax = plt.subplots(figsize=(7.4, 5.4), constrained_layout=True)
scatter_fit(
    ax,
    rmse_ep_all,
    "EP100_all_waves",
    "RMSE_DU",
    title="Cleaned data: O3 RMSE vs early winter all-wave EP flux",
    xlabel="Mean 100 hPa -EP2, 40-80N, Jan20-Feb10",
    ylabel="O3 RMSE, 60-90N 30-70 hPa, Jan1-May30 (DU)",
    color="dodgerblue",
)
savefig(fig, f"{CASE}_reproduce_RMSE_vs_allwave_EPFlux")
plt.show()

## 图：EPFlux 分波贡献

**做什么**：比较 all waves、wave1、wave2、wave rest、wave1+wave2 与 O3 RMSE 的关系。

**怎么做**：所有 EP 指标均为 `mean(-ep2)`，100 hPa、40-80N、Jan20-Feb10；这里比较的是垂直 EP flux 分量，不是 divergence。

**科学问题**：可预报性信号主要来自行星波 1、波 2、synoptic/rest，还是 all-wave 的综合效应？

**预期**：如果长波是主要来源，wave1+wave2 应接近 all waves；如果 synoptic/rest 重要，rest 或 all waves 会明显更强。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("02_wave_component_contribution")


In [ ]:
# -----------------------------
# Which EP-flux wave component matters most?
# EP100_* below is a scalar member metric: mean -ep2 at 100 hPa, 40-80N, Jan20-Feb10.
# It is the vertical EP-flux component with sign flipped so positive follows the old upward-Fz convention;
# it is not div1/div2 or EP-flux divergence.
# -----------------------------
rmse_ep_all = rmse_ep_all.copy()
rmse_ep_all["EP100_wave1_plus_wave2"] = rmse_ep_all["EP100_wave1"] + rmse_ep_all["EP100_wave2"]

EP_SCALAR_COLS = [
    ("EP100_all_waves", "all_waves", "All", "black"),
    ("EP100_wave1", "wave1", "W1", "tab:blue"),
    ("EP100_wave2", "wave2", "W2", "tab:orange"),
    ("EP100_wave_rest", "wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "wave1_plus_wave2", "W1+W2", "tab:purple"),
]

summary_rows = []
for col, key, label, _ in EP_SCALAR_COLS:
    r, p = finite_corr(rmse_ep_all[col], rmse_ep_all["RMSE_DU"])
    summary_rows.append({"metric": key, "label": label, "R_vs_RMSE": r, "P": p, "definition": EP_SCALAR_DESCRIPTION})
r_all_combo, p_all_combo = finite_corr(rmse_ep_all["EP100_wave1_plus_wave2"], rmse_ep_all["EP100_all_waves"])
summary_rows.append({
    "metric": "all_waves_vs_wave1_plus_wave2",
    "label": "All vs W1+W2",
    "R_vs_RMSE": np.nan,
    "P": np.nan,
    "R_aux": r_all_combo,
    "P_aux": p_all_combo,
    "definition": EP_SCALAR_DESCRIPTION,
})
wave_corr = pd.DataFrame(summary_rows)
wave_corr.to_csv(TABLE_DIR / f"{CASE}_wave_component_correlations.csv", index=False)
print(wave_corr)

fig, axes = plt.subplots(2, 3, figsize=(15.8, 8.8), constrained_layout=True)
plot_specs = [
    ("EP100_all_waves", "RMSE_DU", "All vs RMSE", "EP100 all", "O3 RMSE (DU)", "black"),
    ("EP100_wave1", "RMSE_DU", "W1 vs RMSE", "EP100 W1", "O3 RMSE (DU)", "tab:blue"),
    ("EP100_wave2", "RMSE_DU", "W2 vs RMSE", "EP100 W2", "O3 RMSE (DU)", "tab:orange"),
    ("EP100_wave_rest", "RMSE_DU", "Rest vs RMSE", "EP100 rest", "O3 RMSE (DU)", "tab:green"),
    ("EP100_wave1_plus_wave2", "RMSE_DU", "W1+W2 vs RMSE", "EP100 W1+W2", "O3 RMSE (DU)", "tab:purple"),
    ("EP100_wave1_plus_wave2", "EP100_all_waves", "All vs W1+W2", "EP100 W1+W2", "EP100 all", "0.25"),
]
for ax, (xcol, ycol, title, xlabel, ylabel, color) in zip(axes.ravel(), plot_specs):
    scatter_fit(ax, rmse_ep_all, xcol, ycol, title=title, xlabel=xlabel, ylabel=ylabel, color=color, annotate_members=False)
fig.suptitle("EP100 = mean(-ep2), 100 hPa, 40-80N, Jan20-Feb10", fontsize=11)
savefig(fig, f"{CASE}_RMSE_vs_EPFlux_by_wave")
plt.show()


## 图：spread timing：O3、EPFlux 与 U60N10

**做什么**：单独检查 ensemble spread 出现的时间顺序，比较 O3、各分波 EPFlux、以及 60N 10 hPa zonal-mean wind 的 spread 是否同步或谁领先。

**怎么做**：O3 使用 `60-90N, 30-70 hPa` partial O3；EPFlux 使用 `EP100 = mean(-ep2)`，即 100 hPa、40-80N；U60N10 从 raw U 插值到 10 hPa、取 60N 最近纬度并做 zonal mean。所有曲线都画相对 lead day 0 的 spread change，并按各自时间序列标准化，因此曲线从 0 开始，只比较时序和相对增长，不比较物理量单位大小。

**科学问题**：EPFlux spread 是否早于 O3 spread？极夜急流 spread 是否与 EPFlux spread 同步，提示动力过程先于臭氧响应？

**预期**：如果早期波活动是臭氧可预报性差异的源头，EPFlux spread 应早于或至少不晚于 O3 spread；U60N10 spread 可能在波活动增强后随之扩大。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("03_spread_timing_o3_epflux_u60n10")


In [ ]:
# -----------------------------
# Spread timing: does EP-flux spread lead ozone and vortex-wind spread?
# O3: cleaned partial_O3, 60-90N, 30-70 hPa.
# EP100 components: -ep2 at 100 hPa, 40-80N.
# U60N10: raw U interpolated to 10 hPa, nearest 60N, zonal mean.
# All curves are spread changes relative to lead day 0 and standardized separately.
# -----------------------------

def zero_start_standardized(values):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0 or not np.isfinite(arr[0]):
        return arr * np.nan
    delta = arr - arr[0]
    scale = np.nanstd(delta[1:])
    if not np.isfinite(scale) or scale == 0:
        scale = 1.0
    return delta / scale


def interp_logp_single_level(da_var: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: float) -> xr.DataArray:
    target = np.array([float(p_tgt_pa)], dtype=float)

    def _interp_col(vcol, pcol):
        vcol = np.asarray(vcol, dtype=float)
        pcol = np.asarray(pcol, dtype=float)
        mask = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if mask.sum() < 2:
            return np.array([np.nan], dtype=float)
        x = np.log(pcol[mask])
        y = vcol[mask]
        order = np.argsort(x)
        return np.interp(np.log(target), x[order], y[order], left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        da_var,
        p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        dask_gufunc_kwargs={"output_sizes": {"plev": 1}},
    )
    return out.assign_coords(plev=target).isel(plev=0, drop=True)


def u60n10_from_file_for_spread(path: Path, start_end=((1, 1), (5, 30))) -> Tuple[xr.DataArray, np.ndarray]:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask).sel(lat=60.0, method="nearest")
        date = date[mask]
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        u = interp_logp_single_level(
            ds["U"].transpose("time", "lon", "lev"),
            p_mid.transpose("time", "lon", "lev"),
            PLEV_U_PA,
        )
        u_zm = u.mean("lon", skipna=True).load()
    u_zm = u_zm.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return u_zm, date


def build_u60n10_case_cache_for_spread(case=CASE, overwrite=False):
    out = CACHE_DIR / f"U60N10_{case}_members.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        date = np.asarray(da["date"].values, dtype=np.int64)
        return da, date
    files = sorted((HINDCAST_ROOT / case / "U").glob("*.U.nc"))
    if not files:
        raise FileNotFoundError(f"No U files for {case}")
    das, mids = [], []
    date0 = None
    for f in files:
        mid = member_short_id(f.name)
        print("U60N10 spread cache", case, mid)
        da, date = u60n10_from_file_for_spread(f)
        das.append(da)
        mids.append(mid)
        date0 = date
    out_da = xr.concat(das, dim=pd.Index(mids, name="member"))
    out_da.name = "U60N10"
    out_da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return out_da, date0

x = np.arange(len(date_h))
fig, ax = plt.subplots(figsize=(11.8, 5.5), constrained_layout=True)

o3_spread = o3_h.std("member", skipna=True).values
ax.plot(x, zero_start_standardized(o3_spread), color="crimson", lw=2.6, label="O3 partial column")

spread_sources = {wave: ep_series[wave].isel(lead_time=slice(0, len(date_h))) for wave in WAVES}
spread_sources["wave1_plus_wave2"] = spread_sources["wave1"] + spread_sources["wave2"]
for wave, da in spread_sources.items():
    spread = da.std("member", skipna=True).values
    ax.plot(x, zero_start_standardized(spread), lw=1.7, label=f"EP100 {WAVE_LABELS[wave]}")

try:
    u60n10_h, u60n10_date = build_u60n10_case_cache_for_spread(CASE)
    n = min(len(date_h), u60n10_h.sizes["lead_time"])
    u_spread = u60n10_h.isel(lead_time=slice(0, n)).std("member", skipna=True).values
    ax.plot(np.arange(n), zero_start_standardized(u_spread), color="0.15", ls="--", lw=2.3, label="U60N10")
except Exception as exc:
    print(f"Skip U60N10 spread curve: {exc}")

mask_ep = date_mask(date_h, start=EP_WINDOW[0], end=EP_WINDOW[1])
if mask_ep.any():
    ax.axvspan(np.where(mask_ep)[0][0], np.where(mask_ep)[0][-1], color="0.85", alpha=0.5, label="EP window")
positions, labels = month_ticks(date_h)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.axhline(0, color="0.35", lw=0.8)
ax.set_ylabel("Spread change from day 0 (standardized)")
ax.set_title("Spread timing: O3, EP100 wave components, and U60N10")
ax.legend(ncol=3, fontsize=8.8)
savefig(fig, f"{CASE}_spread_timing_O3_EPFlux_waves")
plt.show()


## 图：RMSE 时间窗口敏感性

**做什么**：保留原测试，比较 Jan1、Jan20、Feb20 开始计算 O3 RMSE 时，RMSE 与 all-wave EP100 的关系是否稳定。

**怎么做**：EPFlux 固定为 100 hPa、40-80N、Jan20-Feb10 的 all-wave `-ep2`；仅改变 O3 RMSE 的计算起始日期。

**科学问题**：EPFlux-RMSE 关系是否只是由初期 O3 偏差造成，还是对后期误差也有解释力？

**预期**：如果关系稳健，去掉早期 O3 后相关仍应存在。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("04_rmse_window_sensitivity")


In [ ]:
# -----------------------------
# RMSE-window sensitivity: whole window vs Jan20+ vs Feb20+
# -----------------------------
rmse_windows = [
    ("RMSE_Jan01_May30", (1, 1), "Jan1-May30"),
    ("RMSE_Jan20_May30", (1, 20), "Jan20-May30"),
    ("RMSE_Feb20_May30", (2, 20), "Feb20-May30"),
]
merged_frames = []
for col, start, label in rmse_windows:
    df = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=start, end=O3_END, label=label)
    df = df.rename(columns={"RMSE_DU": col})[["member", col, "n_days"]]
    merged_frames.append(df)
rmse_multi = merged_frames[0]
for df in merged_frames[1:]:
    rmse_multi = rmse_multi.merge(df, on="member", how="inner", suffixes=("", "_y"))
rmse_multi = rmse_multi.merge(rmse_ep_all[["member"] + [f"EP100_{w}" for w in WAVES]], on="member", how="left")
rmse_multi.to_csv(TABLE_DIR / f"{CASE}_rmse_window_sensitivity.csv", index=False)
print(rmse_multi.head())

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.9), constrained_layout=True, sharex=True)
for ax, (col, _, label) in zip(axes, rmse_windows):
    dfp = rmse_multi.rename(columns={col: "RMSE_window"})
    scatter_fit(
        ax, dfp, "EP100_all_waves", "RMSE_window",
        title=label,
        xlabel="Mean 100 hPa all-wave -EP2, Jan20-Feb10",
        ylabel="O3 RMSE (DU)",
        color="tab:purple",
        annotate_members=False,
    )
savefig(fig, f"{CASE}_RMSE_window_sensitivity_vs_allwave_EPFlux")
plt.show()